In [1]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\Swamy\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [5]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
  
# data (as pandas dataframes) 
X = breast_cancer_wisconsin_diagnostic.data.features 
y = breast_cancer_wisconsin_diagnostic.data.targets 
  
# metadata 
print(breast_cancer_wisconsin_diagnostic.metadata) 
  
# variable information 
print(breast_cancer_wisconsin_diagnostic.variables) 



{'uci_id': 17, 'name': 'Breast Cancer Wisconsin (Diagnostic)', 'repository_url': 'https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic', 'data_url': 'https://archive.ics.uci.edu/static/public/17/data.csv', 'abstract': 'Diagnostic Wisconsin Breast Cancer Database.', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 569, 'num_features': 30, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Diagnosis'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1993, 'last_updated': 'Fri Nov 03 2023', 'dataset_doi': '10.24432/C5DW2B', 'creators': ['William Wolberg', 'Olvi Mangasarian', 'Nick Street', 'W. Street'], 'intro_paper': {'ID': 230, 'type': 'NATIVE', 'title': 'Nuclear feature extraction for breast tumor diagnosis', 'authors': 'W. Street, W. Wolberg, O. Mangasarian', 'venue': 'Electronic imaging', 'year': 1993, 'journal': None, 'DOI': '1

Splitting and Tarining dataset

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

# X contains the features; y contains the Diagnosis column
y = breast_cancer_wisconsin_diagnostic.data.targets.iloc[:, 0]

print("Feature shape:", X.shape)
print("Class counts:\n", y.value_counts())

# Keep the class proportions similar in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)
    ),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Gaussian Naive Bayes": GaussianNB(),
    "SVM": make_pipeline(
        StandardScaler(), SVC(probability=True, random_state=42)
    ),
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    # Treat malignant (M) as the positive class for AUC
    malignant_index = list(model.classes_).index("M")
    malignant_scores = model.predict_proba(X_test)[:, malignant_index]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "AUC": roc_auc_score(y_test == "M", malignant_scores),
        "Precision": precision_score(
            y_test, predictions, pos_label="M", zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, pos_label="M", zero_division=0
        ),
        "F1": f1_score(y_test, predictions, pos_label="M", zero_division=0),
        "MCC": matthews_corrcoef(y_test, predictions),
    })

results_df = pd.DataFrame(results).set_index("Model")
print(results_df.round(3))

Feature shape: (569, 30)
Class counts:
 Diagnosis
B    357
M    212
Name: count, dtype: int64
                      Accuracy    AUC  Precision  Recall     F1    MCC
Model                                                                 
Logistic Regression      0.965  0.996      0.975   0.929  0.951  0.925
Decision Tree            0.930  0.925      0.905   0.905  0.905  0.849
KNN                      0.956  0.982      0.974   0.905  0.938  0.906
Gaussian Naive Bayes     0.939  0.993      1.000   0.833  0.909  0.871
SVM                      0.974  0.995      1.000   0.929  0.963  0.944


C:\Users\Swamy\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Preparing app.py for loading models and displaying evaluation metrics

In [7]:
from pathlib import Path
import joblib

artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

bundle = {
    "models": models,
    "feature_names": X.columns.tolist(),
}
joblib.dump(bundle, artifact_dir / "model_bundle.joblib")
results_df.to_csv(
    artifact_dir / "model_metrics.csv",
    index=True,
    index_label="Model",
)

print("Saved model bundle and metrics in:", artifact_dir.resolve())

Saved model bundle and metrics in: C:\Users\Swamy\Documents\Sowjanya\AI_ML_WILP\ML_Assignment_2\Webinar 3_Supervised_learning\ML_Assigment_BC\ml_assignment_2\artifacts
